# Mobile-Wallet Fraud Detection

## 1. Problem framing

**Problem statement.** 
Mobile-money providers (such as Vodafone Cash or InstaPay) process millions of transfers, and a small share of them are fraud. The fraud team must decide, for every transaction, whether to block it, flag it for review, or let it through.

**Target variable.** 
`isFraud`: `1` = fraudulent transaction, `0` = honest transaction.

**Main metric and why.** 
Fraud is extremely rare, so accuracy is misleading:
a model that predicts "not fraud" for everything would score above 99% while
catching nothing. I will use **recall** (how much fraud we catch), **precision**
(how many of our alerts are real) and **PR-AUC** (the trade-off between them
across all thresholds) as the main metrics.

**Business trade-off.** A missed fraud costs the customer and the provider the
stolen money and trust, while a false alarm blocks an honest user and creates
support cost. The final threshold will be chosen using a cost matrix (Step 8).

> Note: the exact fraud rate and class counts will be measured in Step 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/Users/omarelmashat/Desktop/NextGen/Mobile Wallet Fraud Detection/data/raw/paysim.csv')

In [ ]:
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.describe().round(2)

In [ ]:
counts = df["isFraud"].value_counts()
print(counts)
print((df['isFraud'].value_counts(normalize=True) * 100).round(4))

In [ ]:
lazy_accuracy = counts[0] / len(df)
print(f"A model that always say 'not fraud' gets {lazy_accuracy:.4%} accuracy.")

In [ ]:
print(pd.crosstab(df['type'], df['isFraud']))

## 2. Data understanding

**Source.** PaySim Mobile Money Simulation, Kaggle
(https://www.kaggle.com/datasets/ealaxi/paysim1). The data is **simulated** from
the logs of a real mobile-money service, so patterns are realistic while no
real customer data is exposed.

**Size.** ___ rows and ___ columns, covering 743 hourly steps (about 30 days).

**Column dictionary**

| Column | Type | Meaning |
|---|---|---|
| `step` | int | Time unit: 1 step = 1 hour |
| `type` | text | PAYMENT, TRANSFER, CASH_OUT, CASH_IN or DEBIT |
| `amount` | float | Transaction amount |
| `nameOrig` | text | ID of the customer who started the transaction |
| `oldbalanceOrg` | float | Sender's balance before |
| `newbalanceOrig` | float | Sender's balance after |
| `nameDest` | text | ID of the receiver |
| `oldbalanceDest` | float | Receiver's balance before |
| `newbalanceDest` | float | Receiver's balance after |
| `isFraud` | int | **Target:** 1 = fraud, 0 = honest |
| `isFlaggedFraud` | int | Flag from the simulator's own rule (leakage risk, see Step 4) |

**Data quality.** Missing values: 0. Duplicate rows: 0.

**Class balance.** Fraud is 8213 of 6.3 million transactions (0.13%), roughly 1 in 770.
A model that always predicts "not fraud" would reach 99.8709% accuracy while
catching no fraud, so accuracy cannot be the main metric.

**Where fraud happens.** Fraud appears only in the transaction types:
CASH_OUT and TRANSFER (all other types have zero fraud). This will guide the modeling choices.

In [ ]:
fraud_rows = df[df['isFraud'] == 1]
honest_rows = df[df['isFraud'] == 0].sample(200_000,random_state=42)
plot_df = pd.concat([fraud_rows, honest_rows])

How many transactions of each type?

In [ ]:
sns.countplot(data=df, x='type', order=df['type'].value_counts().index)
plt.title('Number of Transactions by Type')
plt.show()

How are amounts distributed?

In [ ]:
sns.histplot(np.log10(plot_df['amount'] + 1), bins=50)
plt.xlabel('Log10(amount + 1)')
plt.title('Distribution of Transaction Amounts')
plt.show()

Fraud rate by type (exact, on the full data)

In [ ]:
by_type = df.groupby('type')['isFraud'].agg(count='count', frauds='sum', rate='mean')
by_type['rate_%'] = (by_type['rate'] *100).round(3)
print(by_type)

Are fraudulent amounts different?

In [ ]:
sns.boxplot(data=plot_df, x='isFraud', y = np.log10(plot_df['amount'] + 1))
plt.title('Amount (log scale) for honest vs Fraud')
plt.show()

Does fraud depend on the time of day?

In [ ]:
df['hour'] = df['step'] % 24
hourly = df.groupby('hour')['isFraud'].mean() * 100
hourly.plot(marker='o')
plt.ylabel('fraud rate (%)')
plt.title('Fraud rate by hour of day')
plt.show()

Does fraud empty the sender’s account?

In [ ]:
df['origEmptied'] = ((df['oldbalanceOrg'] > 0) &(df['newbalanceOrig'] == 0))
print(df.groupby('isFraud')['origEmptied'].mean().round(3))

Does the money add up?

In [ ]:
df['errorOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig'] - df['amount']
print(df.groupby('isFraud')['errorOrig'].describe().round(2))

Correlation heatmap

In [ ]:
num_cols = ["step", "amount", "oldbalanceOrg", "newbalanceOrig","oldbalanceDest", "newbalanceDest", "isFraud"]
sns.heatmap(plot_df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation (plotting sample)")
plt.show()

## 3. EDA: key insights

**Univariate**
- Amounts span many orders of magnitude, so a log scale is needed. On that
  scale most transactions sit roughly between 10^3 and 10^6 EGP, with very
  small and very large ones being rare.
- By volume, CASH_OUT (2.24M) and PAYMENT (2.15M) are the most common types;
  TRANSFER (0.53M) and DEBIT (41K) are far less common.

**Insight 1: Fraud lives in only two transaction types.**
All 8,213 frauds are TRANSFER (4,097) or CASH_OUT (4,116), split almost
evenly. CASH_IN, DEBIT and PAYMENT have zero fraud. The model can therefore
focus on these two types, where fraud is 0.296% of 2.77M transactions.

**Insight 2: TRANSFER is the riskiest type per transaction.**
The fraud rate is 0.769% for TRANSFER (about 1 in 130) versus 0.184% for
CASH_OUT (about 1 in 544), roughly 4 times higher.

**Insight 3: Fraud is extremely rare overall.**
Fraud is 0.129% of all transactions (8,213 of 6,362,620), about 1 in 775.
This confirms accuracy is the wrong metric and PR-AUC/recall are needed.

**Insight 4: Fraud drains the sender's account.**
In 97.6% of fraud cases the sender's balance ends at zero, versus 23.8% of
honest transactions, about 4 times more often. This is a strong feature
candidate.

**Insight 5: Balance arithmetic differs between fraud and honest.**
Computing `oldbalanceOrg - newbalanceOrig - amount`, at least 75% of fraud
rows give exactly 0 (the sender is debited exactly the amount), while at
least 75% of honest rows give a negative value. Part of this is because
honest CASH_IN and PAYMENT rows move balances differently, so I will
re-check it within each type in Step 4.

**Other observations**
- Amount vs fraud (boxplot): fraudulent transactions are much larger. The
  median fraud amount is roughly 6 times the median honest amount (about
  10^5.65 vs 10^4.85 EGP on the log scale), and fraud rarely involves small
  amounts. Part of this gap may come from comparing against all honest
  types, so it will be re-checked within TRANSFER and CASH_OUT.
- Fraud by hour of day: the fraud rate spikes in the early hours (peak
  around hours 4 to 5, above 20%) and falls close to zero after hour 9.
  Because honest activity is low at night, this may partly reflect fewer
  honest transactions rather than more fraud. `hour` will be tested as a
  feature.
- Correlation heatmap (plotting sample, rough guide only): `amount` has the
  strongest link with `isFraud` (0.31), then `step` (0.16). The balance
  columns are weak on their own (|r| ≤ 0.06). `oldbalanceOrg`/`newbalanceOrig`
  (0.98) and `oldbalanceDest`/`newbalanceDest` (0.97) are almost duplicates.

**Hypotheses to test in modeling**
1. Single columns explain little, so engineered balance-change features
   (such as "emptied account" and the balance error) should help more.
2. A model trained only on TRANSFER and CASH_OUT should lose little and
   reduce noise.
3. Redundant balance pairs may confuse linear models, so tree-based models
   may do better.

Checking the isFlaggedFraud trap

In [ ]:
print(pd.crosstab(df['isFlaggedFraud'], df['isFraud']))

Filtering and choose the columns

In [ ]:
model_df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
model_df['hour'] = model_df['step'] % 24

feature_cols = ['type', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'hour']
X = model_df[feature_cols]
y = model_df['isFraud']

print("Rows:", X.shape[0], "| frauds:", y.sum(), "| fraud rate:", round(y.mean() * 100, 3))

Splitting before fitting anything

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train frauds:", y_train.sum(), "| Test frauds:", y_test.sum())

putting the feature code in its own file (src/features.py) and importing it

In [ ]:
import sys
sys.path.append('/Users/omarelmashat/Desktop/NextGen/Mobile Wallet Fraud Detection')
from src.features import add_features

building the preprocessing pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

log_scale = Pipeline([
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler()),
])

columns = ColumnTransformer([
    ('num', log_scale, ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'amount_to_balance']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['type']),
    ('keep', 'passthrough', ['hour', 'drains_account', 'dest_was_empty']),
])

preprocess = Pipeline([
    ('features', FunctionTransformer(add_features)),
    ('columns', columns),
])

fitting on train only and inspect

In [ ]:
preprocess.fit(X_train)
X_train_t = preprocess.transform(X_train.head(5))
print(X_train_t.shape)
print(np.round(X_train_t[:, :4].astype(float), 2))
print(X_train_t[:, 4:])

## 4. Preprocessing and leakage control

**Leakage check: `isFlaggedFraud`.** The simulator's own rule flagged only
16 of the 8,213 frauds (0.19%) and had no false alarms. It is useless as a
detector and comes from the simulator's logic rather than the real world, so
it is **dropped**.

**Decisions**

| Item | Decision | Reason |
|---|---|---|
| `isFlaggedFraud` | Drop | Simulator rule, not real-world information |
| `nameOrig`, `nameDest` | Drop | Near-unique IDs would be memorised, not learned |
| PAYMENT, CASH_IN, DEBIT | Filter out | Zero fraud in the data (Step 3); in production they would pass by rule |
| `newbalanceOrig`, `newbalanceDest` | Not used | Only known after the transaction is processed, so unavailable at decision time |
| Missing values | None found | Zero balances are real values, so they are kept |
| Outliers | Keep, apply log transform | Extreme amounts carry the fraud signal |
| `type` | One-hot encoding | Models need numbers |
| Numeric columns | `log1p` then `StandardScaler` | Strong right skew; scaling helps linear models |

**Modeling data.** After keeping only TRANSFER and CASH_OUT: 2,770,409 rows,
8,213 frauds (0.296%).

**Split.** Stratified 80/20 split with `random_state=42`, done **before any
fitting**: 6,570 frauds in the training set and 1,643 in the test set. The
test set stays untouched until the final evaluation.

**Engineered features** (all known at decision time, built in
`src/features.py`):
- `hour`: `step % 24` (fraud spikes in the early hours)
- `drains_account`: the amount is at least the sender's whole balance
- `amount_to_balance`: amount divided by the sender's balance
- `dest_was_empty`: the receiver's balance was 0 before the transaction

**Pipeline.** One scikit-learn `Pipeline`: feature creation, then a
`ColumnTransformer` (log and scale for numeric columns, one-hot for `type`,
passthrough for the rest). Nothing is fitted until training, so scaling
statistics come only from training data. The transformed matrix has 9
columns.

**Planned comparison.** In Step 5 I will also train a version that uses the
"balance after" columns to measure how much they inflate the score. This
shows why they are excluded from the final model.

making the development sample

In [ ]:
from sklearn.model_selection import train_test_split

X_dev, _, y_dev, _ = train_test_split(X_train, y_train, train_size=0.2, random_state=42, stratify=y_train)

print("Dev rows:", len(X_dev), "| frauds:" , y_dev.sum())

baseline   

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

baseline = DummyClassifier(strategy='prior')
scores = cross_val_score(baseline, X_dev, y_dev, cv=cv, scoring='average_precision')
print("Baseline PR-AUC:", scores.mean().round(4))

Logistic regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
scores = cross_val_score(logreg, X_dev, y_dev, cv=cv, scoring='average_precision')
print("Logistic Regression PR-AUC:", scores.mean().round(4), scores.round(4))

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = Pipeline([
    ('prep', preprocess),
    ('model', RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=5, class_weight='balanced_subsample', n_jobs=1 ,random_state=42)),
])
scores = cross_val_score(forest, X_dev, y_dev, cv=cv, scoring='average_precision')
print("Random forest PR-AUC:", scores.mean().round(4), scores.round(4))

Gradient boosting

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

boost = Pipeline([
    ('prep', preprocess),
    ('model', HistGradientBoostingClassifier(random_state=42)),
])
scores = cross_val_score(boost, X_dev, y_dev, cv=cv, scoring='average_precision')
print("Gradient Boosting PR-AUC:", scores.mean().round(4), scores.round(4))

leakage experiment

In [ ]:
d = model_df.loc[X_dev.index].copy()
d['istransfer'] = (d['type'] == 'TRANSFER').astype(int)
d['errorOrig'] = d['oldbalanceOrg'] - d['newbalanceOrig'] - d['amount']

fair_cols = ['istransfer', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'hour']
leaky_cols = fair_cols + ['errorOrig', 'newbalanceOrig', 'newbalanceDest']

for name, cols in [('fair (decision-time only)', fair_cols), ('with balance-after columns', leaky_cols)]:
    model = HistGradientBoostingClassifier(random_state=42)
    s = cross_val_score(model, d[cols], y_dev, cv=cv, scoring='average_precision')
    print(f"{name} PR-AUC:", s.mean().round(4), s.round(4))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=42)

Xd = X_dev.copy()
Xd["is_transfer"] = (Xd["type"] == "TRANSFER").astype(int)
raw = ["amount", "oldbalanceOrg", "oldbalanceDest", "hour", "is_transfer"]

s_raw = cross_val_score(rf, Xd[raw], y_dev, cv=cv, scoring="average_precision")
print("Random forest, raw columns only:", s_raw.mean().round(4))

## 5. Modeling

**Setup.** To compare models quickly, I used a stratified 20% slice of the
training set (443,265 rows, 1,314 frauds) with 3-fold stratified
cross-validation. The test set was not touched. Every model is wrapped in
the same preprocessing pipeline, so scaling statistics are learned inside
each training fold only. The metric is PR-AUC (`average_precision`).
Imbalance is handled with class weights where the algorithm supports them.

| Model | PR-AUC (mean) | Fold scores |
|---|---|---|
| Baseline (predicts the fraud rate) | 0.003 | n/a |
| Logistic regression (balanced weights) | 0.579 | 0.670, 0.394, 0.674 |
| Gradient boosting (default settings) | 0.876 | 0.934, 0.816, 0.877 |
| **Random forest (balanced weights)** | **0.998** | 0.997, 1.000, 0.996 |

**Findings**
- Every model beats the baseline by a wide margin.
- Logistic regression is weakest and unstable across folds: a linear
  boundary cannot express rules such as "amount is at least the balance".
- The random forest is the best and most stable model, so it is the
  candidate for tuning in Step 6.
- Feature engineering gave the largest gain: the same random forest rises
  from 0.817 (raw columns) to 0.998 (with `drains_account`,
  `amount_to_balance` and `dest_was_empty`).

**Leakage experiment.** Using the same rows and model (gradient boosting),
the five raw decision-time columns give PR-AUC 0.583. Adding the
"balance after" columns raises it to 0.933. Those columns are only known
after a transaction is processed, so they are excluded from the final model.
The engineered features (`drains_account`, `amount_to_balance`,
`dest_was_empty`) reach even higher scores while using only information
available at decision time.
- Random forest on the five raw columns only: PR-AUC 0.817, versus 0.998 with
  the engineered features. The three engineered features add about 0.18
  PR-AUC using only information available at decision time, so feature
  engineering matters more than the choice of algorithm.

**Caution.** Scores near 1.0 are unusual. PaySim is simulated, and its
fraud follows a simple pattern (the account is emptied in one transfer), so
real-world performance would likely be lower. This will be stated as a
limitation in the business insight section.

tuning the random forest

In [ ]:
import time
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

forest_pipe = Pipeline([
    ('prep', preprocess),
    ('model', RandomForestClassifier(class_weight='balanced_subsample', n_jobs=-1, random_state=42)),
])

forest_params = {
    'model__n_estimators': [100,200,300],
    'model__max_depth': [8,12,16,20],
    'model__min_samples_leaf': [1,2,5,10,20],
    'model__max_features': ['sqrt', 0.5, 1],
}

t0 = time.time()
forest_search = RandomizedSearchCV(forest_pipe, forest_params, n_iter=8, cv=cv, scoring='average_precision',verbose=1, random_state=42)
forest_search.fit(X_dev, y_dev)
print("Minutes:", round((time.time() - t0) / 60, 1))
print("Best params:", forest_search.best_params_)
print("Best CV PR-AUC:", forest_search.best_score_.round(4))

tuning the gradient boosting

In [ ]:
boost_pipe = Pipeline([
    ("prep", preprocess),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])

boost_params = {
    "model__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "model__max_iter": [100, 200, 300],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [20, 50, 100],
    "model__l2_regularization": [0.0, 0.1, 1.0],
}

t0 = time.time()
boost_search = RandomizedSearchCV(boost_pipe, boost_params, n_iter=12, cv=cv, scoring="average_precision", random_state=42, verbose=1)
boost_search.fit(X_dev, y_dev)
print("Minutes:", round((time.time() - t0) / 60, 1))
print("Best params:", boost_search.best_params_)
print("Best CV PR-AUC:", round(boost_search.best_score_, 4))

looking inside the search results

In [ ]:
def top_results(search, n=5):
    results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
    results['params'] = results['params'].apply(lambda d: {k.replace("model__", ""): v for k, v in d.items()})
    return results[['rank_test_score', 'mean_test_score', 'std_test_score', 'params']].head(n)

pd.set_option('display.max_colwidth', 200)
print(top_results(forest_search))
print(top_results(boost_search))

deciding the model winner

In [ ]:
summary = pd.DataFrame({
    "model": ["Random forest (default)", "Random forest (tuned)",
              "Gradient boosting (default)", "Gradient boosting (tuned)"],
    "cv_pr_auc": [0.9978, forest_search.best_score_, 0.8758, boost_search.best_score_],
}).round(4)
print(summary)

training the winner on the full training set

In [ ]:
from sklearn.base import clone

winner = clone(forest_search.best_estimator_)   
t0 = time.time()
winner.fit(X_train, y_train)
print("Trained on", len(X_train), "rows in", round((time.time() - t0) / 60, 1), "minutes")

## 6. Tuning and validation

**Method.** Both searches used `RandomizedSearchCV` on the 20% development
sample (443,265 rows) with 3-fold stratified cross-validation, scored by
PR-AUC. The test set was not used. Every candidate runs inside the full
preprocessing pipeline, so scaling is learned within each training fold only.

**Random forest** (8 random combinations, 24 fits, 2.2 minutes)
- Best settings: `n_estimators=200`, `max_depth=20`, `min_samples_leaf=10`,
  `max_features="sqrt"`
- Best CV PR-AUC: **0.9985** (fold standard deviation 0.0016)
- The top 5 combinations all scored between 0.9977 and 0.9985, which is
  within one fold standard deviation of each other. The forest is robust to
  its settings.

**Gradient boosting** (12 random combinations, 36 fits, 0.6 minutes)
- Best settings: `learning_rate=0.03`, `max_iter=200`, `max_leaf_nodes=15`,
  `min_samples_leaf=20`, `l2_regularization=0.0`
- Best CV PR-AUC: **0.9913** (fold standard deviation 0.0060)
- Top 5 combinations: 0.9905 to 0.9913.

| Model | CV PR-AUC | Fold std |
|---|---|---|
| Random forest, default | 0.9978 | n/a |
| **Random forest, tuned** | **0.9985** | 0.0016 |
| Gradient boosting, default | 0.8758 | n/a |
| Gradient boosting, tuned | 0.9913 | 0.0060 |

**Findings**
- Tuning transformed gradient boosting (0.876 to 0.991) but barely changed
  the random forest (0.9978 to 0.9985, a gain smaller than the fold-to-fold
  spread).
- The tuned forest is chosen as the final model: its mean score is higher
  and its folds are about four times steadier than boosting (std 0.0016
  versus 0.0060).
- Choosing the best of several candidates slightly inflates the CV score, so
  the honest estimate comes only from the untouched test set in Step 7.

**Final model.** The tuned random forest was retrained on the **full
training set** (2,216,327 rows) in 1.4 minutes.

ranking metrics

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

proba = winner.predict_proba(X_test)[:, 1]

print("Test PR-AUC :", round(average_precision_score(y_test, proba), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
print("Baseline PR-AUC (= fraud rate):", round(y_test.mean(), 4))

hard decisions at the default threshold

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred = (proba >= 0.5).astype(int)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["honest", "fraud"], digits=4))

precision-recall curve

In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, proba)
p05 = precision_score = (pred[y_test == 1].sum() / max(pred.sum(), 1))   # precision at 0.5
r05 = pred[y_test == 1].sum() / y_test.sum()                              # recall at 0.5

plt.plot(rec, prec, label="Random forest")
plt.scatter([r05], [p05], color="red", zorder=3, label="threshold 0.5")
plt.axhline(y_test.mean(), color="gray", linestyle="--", label="baseline")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-recall curve (test set)")
plt.legend(); plt.show()

error analysis

In [ ]:
res = add_features(X_test)
res["y_true"] = y_test.values
res["proba"] = proba
res["outcome"] = np.select(
    [(res.y_true == 1) & (pred == 1), (res.y_true == 1) & (pred == 0),
     (res.y_true == 0) & (pred == 1)],
    ["TP", "FN", "FP"], default="TN")

print(res["outcome"].value_counts())
print()
print(pd.crosstab(res["type"], res["outcome"]))
print()
print(res[res["outcome"] != "TN"].groupby("outcome")[
    ["amount", "oldbalanceOrg", "drains_account", "dest_was_empty", "hour"]].median().round(2))

seeing how close the misses were

In [ ]:
fn = res[res["outcome"] == "FN"]
print("Missed frauds:", len(fn))
print(fn["proba"].describe().round(3))

## 7. Evaluation on the untouched test set

The final model (tuned random forest, trained on the full training set) was
scored once on the test set: 554,082 transactions, 1,643 of them fraud.

| Metric | Value |
|---|---|
| PR-AUC | **0.998** (baseline: 0.003) |
| ROC-AUC | 0.9991 |
| Precision (fraud, threshold 0.5) | 0.9994 |
| Recall (fraud, threshold 0.5) | 0.9963 |
| F1 (fraud, threshold 0.5) | 0.9979 |

**Confusion matrix (threshold 0.5)**

| | Predicted honest | Predicted fraud |
|---|---|---|
| **Actually honest** | 552,438 | 1 |
| **Actually fraud** | 6 | 1,637 |

**Reading the results**
- The model caught 1,637 of 1,643 frauds (99.6%) and raised only 1 false
  alarm among 552,439 honest transactions.
- The test PR-AUC (0.998) is very close to the cross-validation score
  (0.9985), so there is no sign of overfitting to the training data.
- Accuracy is 99.9987%, but a model that always says "honest" already
  scores 99.70%, so accuracy hardly separates good from useless here.
  PR-AUC, precision and recall carry the real information.
- PR curve: precision stays close to 1.0 up to about 99% recall, so almost
  all frauds can be caught with almost no false alarms. The default
  threshold of 0.5 sits at the top-right corner of the curve (recall 0.996,
  precision 0.999). Precision only falls sharply at the very end, where
  catching the last few hard frauds would require flagging many honest
  transactions.

**Error analysis**

*Missed frauds (6 false negatives):* 3 were CASH_OUT and 3 were TRANSFER, so
the model is not weak on one type. Compared with the caught frauds, the
missed ones break the usual pattern:
- at least 4 of the 6 did **not** empty the sender's account
  (`drains_account` median 0, versus 1 for caught frauds);
- at least 4 of the 6 went to a destination account that was **not** empty
  (`dest_was_empty` median 0, versus 1 for caught frauds);
- their median amount (about 259,000) is lower than that of caught frauds
  (about 452,000), and their median hour is 16, in the daytime.
- Model confidence on the missed frauds: these were mostly confident
  misses, not near misses. The model gave them a mean fraud probability of
  0.086 (median 0.03), and the highest was 0.317, so none came close to the
  0.5 cutoff. Lowering the threshold would therefore recover at most one or
  two of them, and it would also raise false alarms.

The model has learned the simple pattern "fraudster takes the whole balance
and sends it to a fresh account". Frauds that look different slip through,
and the model does not see them as suspicious at all.

*False alarm (1 false positive):* a transfer of about 1.08 million from an
account with a recorded balance of 0 to an empty destination account. It
copies the fraud pattern even though it is labelled honest, so a human
reviewer would plausibly have stopped it as well. With a single case,
nothing more general can be said.

**Limitations of this evaluation**
- With only 6 misses and 1 false alarm, the exact recall and precision are
  not stable estimates; a few extra errors would visibly move them.
- PaySim is simulated, and its fraud is unusually regular, so these scores
  are likely higher than real-world performance would be.
- The data covers about 30 days. Real fraudsters change tactics, so the
  model would need regular retraining and monitoring.

Out-of-fold probabilities on the training set

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.base import clone

t0 = time.time()
oof_proba = cross_val_predict(
    clone(winner), X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]
print("Minutes:", round((time.time() - t0) / 60, 1))
print("OOF PR-AUC:", round(average_precision_score(y_train, oof_proba), 4))

cost of every threshold


In [ ]:
y_tr = y_train.values
amt_tr = X_train["amount"].values
fraud_amt = amt_tr * (y_tr == 1)      # amount if fraud, else 0
honest = (y_tr == 0)

thresholds = np.round(np.arange(0.01, 1.0, 0.01), 2)

def cost_curve(proba, c_fp):
    rows = []
    for t in thresholds:
        flag = proba >= t
        missed = fraud_amt[~flag].sum() # not flagged
        false_alarms = (honest & flag).sum()
        rows.append((t, missed, false_alarms, missed + c_fp * false_alarms))
    return pd.DataFrame(rows, columns=["threshold", "missed_fraud_amount", "false_alarms", "total_cost"])

C_FP = 100   # ASSUMPTION: cost of one false alarm (currency units)
curve = cost_curve(oof_proba, C_FP)

best = curve.loc[curve["total_cost"].idxmin()]
print(best)

cost curve

In [ ]:
plt.plot(curve["threshold"], curve["total_cost"] / 1e6)
plt.axvline(best["threshold"], color="red", linestyle="--", label=f"best = {best['threshold']:.2f}")
plt.axvline(0.5, color="gray", linestyle=":", label="default 0.5")
plt.xlabel("Threshold"); plt.ylabel("Total cost (millions)")
plt.title("Total cost by threshold (out-of-fold, training data)")
plt.legend(); plt.show()

how much does the abswer depened on the false-alarn cost?


In [ ]:
rows = []
for c in [10, 100, 1_000, 10_000, 100_000]:
    cc = cost_curve(oof_proba, c)
    b = cc.loc[cc["total_cost"].idxmin()]
    rows.append((c, b["threshold"], int(b["false_alarms"]), round(b["missed_fraud_amount"] / 1e6, 2)))
sens = pd.DataFrame(rows, columns=["cost_per_false_alarm", "best_threshold", "false_alarms", "missed_amount_millions"])
print(sens)

final report on the test set

In [ ]:
T = float(best["threshold"])

y_te = y_test.values
amt_te = X_test["amount"].values

def evaluate(t):
    flag = proba >= t
    caught = amt_te[(y_te == 1) & flag].sum()
    missed = amt_te[(y_te == 1) & ~flag].sum()
    fa = ((y_te == 0) & flag).sum()
    return caught, missed, fa, missed + C_FP * fa

total_fraud = amt_te[y_te == 1].sum()
print("Total fraud amount in test set (no model):", round(total_fraud / 1e6, 2), "million")
for label, t in [("chosen threshold", T), ("default 0.5", 0.5)]:
    caught, missed, fa, cost = evaluate(t)
    print(f"{label} ({t:.2f}): caught {caught/1e6:.2f}M | missed {missed/1e6:.3f}M | false alarms {fa} | total cost {cost/1e6:.3f}M")

In [136]:
flag = proba >= T
tp = int(((y_te == 1) & flag).sum())
fn = int(((y_te == 1) & ~flag).sum())
fp = int(((y_te == 0) & flag).sum())
print("caught:", tp, "| missed:", fn, "| false alarms:", fp)
print("recall:", round(tp / (tp + fn), 4), "| precision:", round(tp / (tp + fp), 4))

caught: 1640 | missed: 3 | false alarms: 783
recall: 0.9982 | precision: 0.6768


## 8. Business insight

**Cost assumptions.** A missed fraud costs the transaction amount, which
comes from the data. A false alarm (an honest transaction blocked) costs a
fixed **100 currency units**, which is an **assumption**: PaySim contains no
information on support or customer-loss costs, and it does not state a
currency. The sensitivity table below shows how much the answer depends on it.

**Choosing the threshold without touching the test set.** I generated
out-of-fold probabilities on the training set (3 folds, OOF PR-AUC 0.997),
computed the total cost for every threshold from 0.01 to 0.99, and picked the
cheapest one.

- Best threshold: **0.02** (training out-of-fold cost: 6.81M missed fraud
  plus 3,586 false alarms x 100 = 7.17M).
- The cost curve has a long flat valley from about 0.05 to 0.8, and cost
  rises sharply only above about 0.9, where almost nothing is flagged. The
  choice is therefore not fragile in that range.

**Sensitivity to the false-alarm cost (out-of-fold, training data)**

| Cost per false alarm | Best threshold | False alarms | Missed fraud (millions) |
|---|---|---|---|
| 10 | 0.02 | 3,586 | 6.81 |
| 100 | 0.02 | 3,586 | 6.81 |
| 1,000 | 0.03 | 2,525 | 7.03 |
| 10,000 | 0.60 | 3 | 9.83 |
| 100,000 | 0.60 | 3 | 9.83 |

Up to a cost of 1,000 the best threshold stays very low (0.02 to 0.03). Between
1,000 and 10,000 it jumps to 0.60, so the right threshold depends on how
expensive a false alarm really is for the provider.

**Result on the test set (the threshold was fixed before looking at it)**

| | No model | Default 0.5 | Chosen 0.02 |
|---|---|---|---|
| Fraud money caught | 0 | 2,425.67M | **2,426.83M** |
| Fraud money missed | 2,427.47M | 1.799M | **0.640M** |
| False alarms | 0 | 1 | 783 |
| Total cost (100 per false alarm) | 2,427.47M | 1.799M | **0.719M** |

- At 0.02 the model stops **99.97%** of the fraud value in the test set
  (99.93% at the default 0.5).
- Compared with the default threshold, the total cost falls by about 60%
  (1.799M to 0.719M), at the price of 783 extra false alarms: 0.14% of the
  552,439 honest test transactions.

**Recommendation**
1. Deploy the model: it blocks almost all fraud money while flagging a tiny
   share of honest traffic.
2. Use two levels. Block automatically at high scores (0.5 and above, where
   there was 1 false alarm), and send scores between 0.02 and 0.5 to a quick
   manual review or an extra verification step instead of blocking outright.
   This keeps most of the gain without 783 blocked customers.
3. Have the business measure its real cost per false alarm and re-choose the
   threshold with the sensitivity table. The app provides a slider for this.
4. Retrain regularly and monitor the fraud rate, because fraud tactics change.

**Limitations**
- The false-alarm cost is assumed, and the data gives no currency or real
  cost information.
- Missed-fraud money is dominated by a handful of large transactions, so the
  dollar results are noisy: the training estimate (6.81M missed on about
  2.2M rows) and the test result (0.64M on about 0.55M rows) differ by more
  than their sizes alone would explain.
- PaySim is simulated, and its fraud is unusually regular (accounts emptied
  in one transfer), so real-world performance would likely be lower.
- The data covers about 30 days and one environment; the model has not been
  tested against changing fraud behaviour.

In [137]:
import os, sys, json, joblib, sklearn, numpy, pandas
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

# 1) the trained pipeline
joblib.dump(winner, "../models/fraud_pipeline.joblib", compress=3)
size_mb = os.path.getsize("../models/fraud_pipeline.joblib") / 1e6
print("Model file size (MB):", round(size_mb, 1))

# 2) the cost curve (for the slider page)
curve[["threshold", "missed_fraud_amount", "false_alarms"]].to_csv("../models/cost_curve.csv", index=False)

# 3) settings and measured test results
pred05 = (proba >= 0.5).astype(int)
config = {
    "threshold": float(T),
    "c_fp": float(C_FP),
    "test": {
        "rows": int(len(y_test)),
        "frauds": int(y_test.sum()),
        "pr_auc": round(float(average_precision_score(y_test, proba)), 4),
        "roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
        "precision_at_0.5": round(float(precision_score(y_test, pred05)), 4),
        "recall_at_0.5": round(float(recall_score(y_test, pred05)), 4),
    },
}
with open("../models/config.json", "w") as f:
    json.dump(config, f, indent=2)

# 4) a demo CSV for testing the upload (no labels inside)
demo = pd.concat([X_test[y_test == 1].sample(20, random_state=1),
                  X_test[y_test == 0].sample(480, random_state=1)]).sample(frac=1, random_state=1)
demo.to_csv("../reports/sample_transactions.csv", index=False)

# 5) versions, needed for requirements.txt
print("python", sys.version.split()[0], "| scikit-learn", sklearn.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__, "| joblib", joblib.__version__)

Model file size (MB): 1.1
python 3.9.6 | scikit-learn 1.6.1 | numpy 2.0.2 | pandas 2.3.3 | joblib 1.5.3


In [138]:
import subprocess, sys
import pandas as pd

row = pd.DataFrame([{"type": "TRANSFER", "amount": 181000.0,
                     "oldbalanceOrg": 181000.0, "oldbalanceDest": 0.0, "hour": 4}])
print("in this notebook:", round(winner.predict_proba(row)[0, 1], 4))

code = '''
import sys; sys.path.insert(0, "..")
import joblib, pandas as pd
m = joblib.load("../models/fraud_pipeline.joblib")
row = pd.DataFrame([{"type": "TRANSFER", "amount": 181000.0,
                     "oldbalanceOrg": 181000.0, "oldbalanceDest": 0.0, "hour": 4}])
print("fresh process:", round(m.predict_proba(row)[0, 1], 4))
'''
r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])

in this notebook: 1.0
fresh process: 1.0
 
